In [5]:
# Cell 1: Setup project root (fix path)
from pathlib import Path
import os, sys

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD =", os.getcwd())

CWD = D:\logistics_AI


In [6]:
# Cell 2: Config
PROCESS_CODE = "WAREHOUSE_FULFILLMENT"

DATA_DIR = "data/synth_optimal_3process_v1"
REGISTRY_DIR = "registry/synth_optimal_v1"
MODEL_DIR = "model/process_models"

# Train params
N_ESTIMATORS = 300
CONTAMINATION = 0.06
RANDOM_STATE = 42

INCLUDE_CONTEXT_NUMERIC = False

In [7]:
# Cell 3: Sanity check input data
import pandas as pd
from pathlib import Path

events_path = Path(DATA_DIR) / "events.csv"
print("events_path:", events_path, "| exists:", events_path.exists())

df = pd.read_csv(events_path)
print("rows:", len(df))
print("process_code counts (top):")
print(df["process_code"].value_counts().head(10))

assert PROCESS_CODE in set(df["process_code"].unique()), f"Missing process_code in events.csv: {PROCESS_CODE}"

events_path: data\synth_optimal_3process_v1\events.csv | exists: True
rows: 692576
process_code counts (top):
process_code
TRUCKING_DELIVERY_FLOW      348000
WAREHOUSE_FULFILLMENT       276000
IMPORT_CUSTOMS_CLEARANCE     68576
Name: count, dtype: int64


In [8]:
# Cell 4: Train process model (WAREHOUSE)
from api.process_ai.train import train_process_model

summary = train_process_model(
    data_dir=DATA_DIR,
    registry_dir=REGISTRY_DIR,
    model_root_dir=MODEL_DIR,
    process_code=PROCESS_CODE,
    include_context_numeric=INCLUDE_CONTEXT_NUMERIC,
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
)

summary

{'process_code': 'WAREHOUSE_FULFILLMENT',
 'events_rows_used': 276000,
 'cases_used': 12000,
 'include_context_numeric': False,
 'n_estimators': 300,
 'contamination': 0.06,
 'random_state': 42,
 'artifacts_dir': 'model\\process_models\\WAREHOUSE_FULFILLMENT',
 'validation_warnings': [],
 'feature_report': {'cases': 12000,
  'dropped_cases': 0,
  'repeated_case_count': 0,
  'missing_step_cases': 0}}

In [9]:
# Cell 5: Verify artifacts saved
from pathlib import Path

art_dir = Path(MODEL_DIR) / PROCESS_CODE
print("artifacts_dir:", art_dir.resolve())
print("exists:", art_dir.exists())

sorted([p.name for p in art_dir.glob("*")])

artifacts_dir: D:\logistics_AI\model\process_models\WAREHOUSE_FULFILLMENT
exists: True


['baselines.json',
 'feature_schema.json',
 'model.pkl',
 'scaler.pkl',
 'score_quantiles.json',
 'train_summary.json']

In [10]:
# Cell 6: Quick inference smoke test (1 case)
import pandas as pd
from pathlib import Path
from api.process_ai.inference import load_process_artifacts, analyze_case

art = load_process_artifacts(
    process_code=PROCESS_CODE,
    model_root_dir=MODEL_DIR,
    registry_dir=REGISTRY_DIR,
)

df = pd.read_csv(Path(DATA_DIR) / "events.csv")
cid = df.loc[df["process_code"] == PROCESS_CODE, "case_id"].iloc[0]
one = df[(df["process_code"] == PROCESS_CODE) & (df["case_id"] == cid)].copy()

result = analyze_case(one, art, allow_unknown_steps=False)
result

{'ok': True,
 'process_code': 'WAREHOUSE_FULFILLMENT',
 'case_id': 'ORD_5260181590',
 'risk_score': 78.0,
 'raw_anomaly': 0.432292,
 'overall_severity': 'CRITICAL',
 'top_steps': [{'step_code': 'STEP_011_PACKING_START',
   'duration_min': 58.583,
   'p95': 35.201,
   'deviation_factor': 1.664,
   'z_score': 4.134,
   'severity': 'CRITICAL'},
  {'step_code': 'STEP_006_PICK_START',
   'duration_min': 25.167,
   'p95': 24.1,
   'deviation_factor': 1.044,
   'z_score': 1.941,
   'severity': 'WATCHLIST'},
  {'step_code': 'STEP_023_WAREHOUSE_CLOSEOUT',
   'duration_min': 19.983,
   'p95': 19.917,
   'deviation_factor': 1.003,
   'z_score': 1.878,
   'severity': 'WATCHLIST'},
  {'step_code': 'STEP_015_MANIFEST_CREATED',
   'duration_min': 17.467,
   'p95': 19.733,
   'deviation_factor': 0.885,
   'z_score': 1.472,
   'severity': 'NORMAL'},
  {'step_code': 'STEP_013_WEIGH_DIM_CHECK',
   'duration_min': 22.883,
   'p95': 29.334,
   'deviation_factor': 0.78,
   'z_score': 1.08,
   'severity': 'N